In [2]:
import pandas as pd

categorical_cols = [
    "시술 당시 나이", "시술 유형", "배아 생성 주요 이유",
    "난자 출처", "정자 출처", "난자 기증자 나이", "정자 기증자 나이",
    "시술_인공수정방식"
]

train = pd.read_csv('train_preprocessed_ff.csv')
test = pd.read_csv('test_preprocessed_ff.csv')

# Int64 -> 문자열 -> category 순서로 변환 (XGBoost가 카테고리 값으로 문자열을 요구함)
for col in categorical_cols:
    train[col] = train[col].astype('Int64').astype(str).replace('<NA>', pd.NA).astype('category')
    test[col] = test[col].astype('Int64').astype(str).replace('<NA>', pd.NA).astype('category')

print('Train:', train.shape, '/ Test:', test.shape)

Train: (256351, 69) / Test: (90067, 68)


In [3]:
from sklearn.model_selection import train_test_split

target = '임신 성공 여부'
X = train.drop(columns=[target])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [4]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    n_estimators=500
)

model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

valid_pred = model.predict_proba(X_valid)[:, 1]
print('Validation ROC-AUC:', roc_auc_score(y_valid, valid_pred))

Validation ROC-AUC: 0.7242635802970998


In [6]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.head(15))

             feature  importance
2              시술 유형    0.569421
37          이식된 배아 수    0.145357
48             난자 출처    0.032524
39          저장된 배아 수    0.016465
0           시술 당시 나이    0.010908
60         배아 이식 경과일    0.010515
5   착상 전 유전 검사 사용 여부    0.008916
53       신선 배아 사용 여부    0.007326
34         총 생성 배아 수    0.005931
33          DI 출산 횟수    0.005584
49             정자 출처    0.005560
28           총 임신 횟수    0.005141
50         난자 기증자 나이    0.005114
31           총 출산 횟수    0.004939
29         IVF 임신 횟수    0.004573


In [17]:
test_pred = model.predict_proba(test)[:, 1]

test_id = pd.read_csv('test.csv')['ID']  # 원본에서 ID 가져오기

result = pd.DataFrame({'ID': test_id, 'xgb_pred': test_pred})
result.to_csv('xgb_predictions.csv', index=False)
print('저장 완료:', result.shape)

저장 완료: (90067, 2)


In [11]:

valid_pred = model.predict_proba(X_valid)[:, 1]

valid_result = pd.DataFrame({
    'y_valid': y_valid.values,
    'xgb_valid_pred': valid_pred
})
valid_result.to_csv('xgb_valid_predictions.csv', index=False)

In [12]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import xgboost as xgb

param_dist = {
    'max_depth': [4, 5, 6, 7],
    'learning_rate': [0.03, 0.05, 0.1],
    'n_estimators': [200, 300],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
}

base_model = xgb.XGBClassifier(
    objective='binary:logistic', eval_metric='auc',
    enable_categorical=True, tree_method='hist',
    random_state=42,
    n_jobs=-1
)

search = RandomizedSearchCV(
    base_model, param_distributions=param_dist,
    n_iter=6,
    scoring='roc_auc',
    cv=2,
    random_state=42,
    n_jobs=1,
    verbose=3
)
search.fit(X_train, y_train)

Fitting 2 folds for each of 6 candidates, totalling 12 fits
[CV 1/2] END colsample_bytree=1.0, learning_rate=0.1, max_depth=5, n_estimators=300, subsample=0.8;, score=0.737 total time=  11.3s
[CV 2/2] END colsample_bytree=1.0, learning_rate=0.1, max_depth=5, n_estimators=300, subsample=0.8;, score=0.735 total time=  18.2s
[CV 1/2] END colsample_bytree=1.0, learning_rate=0.1, max_depth=7, n_estimators=300, subsample=0.8;, score=0.732 total time=  10.6s
[CV 2/2] END colsample_bytree=1.0, learning_rate=0.1, max_depth=7, n_estimators=300, subsample=0.8;, score=0.730 total time=   8.8s
[CV 1/2] END colsample_bytree=0.9, learning_rate=0.1, max_depth=7, n_estimators=200, subsample=0.8;, score=0.735 total time=   7.8s
[CV 2/2] END colsample_bytree=0.9, learning_rate=0.1, max_depth=7, n_estimators=200, subsample=0.8;, score=0.733 total time=   7.0s
[CV 1/2] END colsample_bytree=1.0, learning_rate=0.05, max_depth=5, n_estimators=300, subsample=0.8;, score=0.739 total time=   8.0s
[CV 2/2] END co

RandomizedSearchCV(cv=2,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=True,
                                           eval_metric='auc',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_constrain...
                                           min_child_weight=None, missing=nan,
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=-1,
                                           num_parallel_tree=None, ...),
                   n_iter=6, n_jobs=1,
                   param_distributions={'colsample_bytree': [0.8, 0.9, 1.0],
                                        'learning_rate': [0.03, 0.05, 0.1],
                                        'max_depth': [4, 5, 6, 7],
                                        'n_estimators': [200, 300],
                                        'subsample': [0.8, 0.9, 1.0]},
                   random_state=42, scoring='roc_auc', verbose=3)

In [18]:
# 1) 최적 모델 확정
best_model = search.best_estimator_
print('최적 파라미터:', search.best_params_)

# 2) 검증 데이터로 최종 AUC 확인
valid_pred = best_model.predict_proba(X_valid)[:, 1]
final_auc = roc_auc_score(y_valid, valid_pred)
print('튜닝 후 검증 AUC:', final_auc)

# 3) test 예측 (앙상블 재료용 - 기존 xgb_predictions.csv 덮어쓰기)
test_pred = best_model.predict_proba(test)[:, 1]
test_id = pd.read_csv('test.csv')['ID']
result = pd.DataFrame({'ID': test_id, 'xgb_pred': test_pred})
result.to_csv('xgb_predictions.csv', index=False)
print('xgb_predictions.csv 갱신 완료:', result.shape)

# 4) valid 예측 (앙상블 AUC 재확인용 - 기존 xgb_valid_predictions.csv 덮어쓰기)
valid_result = pd.DataFrame({'y_valid': y_valid.values, 'xgb_valid_pred': valid_pred})
valid_result.to_csv('xgb_valid_predictions.csv', index=False)
print('xgb_valid_predictions.csv 갱신 완료:', valid_result.shape)

최적 파라미터: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
튜닝 후 검증 AUC: 0.7357580197631766
xgb_predictions.csv 갱신 완료: (90067, 2)
xgb_valid_predictions.csv 갱신 완료: (51271, 2)


In [16]:
test_check = pd.read_csv('test.csv')
print(test_check.shape)          # (90067, 68)이 나와야 정상
print(test_check['ID'].duplicated().sum())  # 중복 ID 있는지 확인
print(test_check.tail(10))       # 마지막 부분에 이상한 값이 있는지 확인

(90067, 68)
0
               ID 시술 시기 코드 시술 당시 나이  임신 시도 또는 마지막 임신 경과 연수 시술 유형   특정 시술 유형  \
90057  TEST_90057   TRDQAZ  만40-42세                    NaN   IVF       ICSI   
90058  TEST_90058   TRZKPL  만38-39세                    NaN   IVF    Unknown   
90059  TEST_90059   TRJXFG  만38-39세                    NaN   IVF       ICSI   
90060  TEST_90060   TRVNRY  만35-37세                    NaN   IVF        IVF   
90061  TEST_90061   TRJXFG  만18-34세                    NaN   IVF       ICSI   
90062  TEST_90062   TRDQAZ  만18-34세                    NaN   IVF  ICSI:ICSI   
90063  TEST_90063   TRYBLT  만43-44세                    NaN   IVF    Unknown   
90064  TEST_90064   TRVNRY  만18-34세                    NaN   IVF        IVF   
90065  TEST_90065   TRCMWS  만43-44세                    NaN   IVF    Unknown   
90066  TEST_90066   TRDQAZ  만18-34세                    NaN   IVF        IVF   

       배란 자극 여부    배란 유도 유형  단일 배아 이식 여부  착상 전 유전 검사 사용 여부  ...  신선 배아 사용 여부  \
90057         1  기록되지 않은 시행         